# Notebook 7: Cluster Evaluation

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 50 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Implement the **silhouette score** from scratch and explain what it measures
2. Implement the **Davies-Bouldin index** from scratch and explain what it measures
3. Understand **inertia (WCSS)** as an internal metric and its limitations
4. Use **external metrics** (ARI, NMI) correctly when ground-truth labels are available
5. Combine multiple metrics to choose the optimal number of clusters *k*

---

**Week 11 Theme:** Customer segmentation / retail data

## The Jigsaw Puzzle Analogy

Evaluating clustering without ground-truth labels is like judging a jigsaw puzzle you assembled **without the box art**.

You cannot look at the picture on the box (labels) to check whether you got it right. But you can still ask two sensible questions:

- **Are pieces that are joined together similar in color and texture?** — If yes, your clusters are tight. This is what the **silhouette score** measures: cohesion within a cluster.
- **Are pieces in one section clearly different from pieces in other sections?** — If yes, your clusters are well separated. This is also part of the silhouette score (separation) and the **Davies-Bouldin index** (which explicitly penalises clusters that look too similar to each other).

A good clustering result has **compact** clusters (similar pieces together) that are **well-separated** from each other.

## Why Does This Matter for ML?

In supervised learning, you evaluate a model by comparing predictions to true labels. **Clustering has no true labels** — that is the whole point of unsupervised learning.

This creates two practical problems:

1. **How do you pick *k*?** K-Means requires you to specify the number of clusters. You need a principled way to decide whether *k = 3* or *k = 5* segments your customers better.
2. **How do you compare algorithms?** Is K-Means better than DBSCAN on this dataset? You cannot just look at accuracy.

Cluster evaluation metrics solve both problems. They are the **only honest way** to tune and compare clustering algorithms without "peeking" at labels that do not exist in real deployments.

**Business context:** In retail customer segmentation, choosing the wrong *k* means you either over-segment (too many tiny groups, impossible to personalise at scale) or under-segment (too few groups, marketing messages are too generic). The metrics in this notebook guide that decision.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from sklearn.preprocessing import StandardScaler

np.random.seed(42)           # reproducibility for all random operations
sns.set_theme(style='whitegrid')  # consistent seaborn plot styling

print('Libraries loaded successfully.')

## Internal vs. External Metrics

Cluster evaluation metrics fall into two camps:

### Internal Metrics
Use **only the data and the cluster assignments** — no ground-truth labels needed.

| Metric | Intuition | Range | Better = |
|--------|-----------|-------|----------|
| **Inertia (WCSS)** | Total squared distance of points to their centroid | 0 → ∞ | Lower |
| **Silhouette Score** | How similar a point is to its own cluster vs. nearest other | −1 → +1 | Higher |
| **Davies-Bouldin Index** | Average worst-case ratio of scatter to separation | 0 → ∞ | Lower |
| **Calinski-Harabasz** | Ratio of between-cluster to within-cluster dispersion | 0 → ∞ | Higher |

### External Metrics
Compare cluster assignments to **ground-truth labels** — only available in research or validation settings.

| Metric | Intuition | Range | Better = |
|--------|-----------|-------|----------|
| **ARI** | Agreement adjusted for chance | −1 → +1 | Higher |
| **NMI** | Mutual information normalised to [0,1] | 0 → 1 | Higher |
| **Homogeneity** | Each cluster contains only one class | 0 → 1 | Higher |

> **Rule of thumb:** In real ML projects, you almost always use internal metrics. External metrics are useful for benchmarking algorithms on datasets where you happen to know the true groupings.

In [ ]:
np.random.seed(42)

# Generate a clean 4-cluster dataset — our running example throughout this notebook
# make_blobs creates Gaussian blobs; cluster_std controls how spread out each blob is
X_blobs, y_true = make_blobs(
    n_samples=400,
    centers=4,
    cluster_std=0.8,   # tight clusters so metrics behave cleanly
    random_state=42
)

# Quick peek at the data
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(X_blobs[:, 0], X_blobs[:, 1],
                     c=y_true, cmap='tab10', s=30, alpha=0.7)
ax.set_title('Running Example: 4-Cluster Blobs (colors = true labels)')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
plt.colorbar(scatter, ax=ax, label='True cluster')
plt.tight_layout()
plt.show()

print(f'Dataset shape: {X_blobs.shape}')
print(f'True cluster counts: {np.bincount(y_true)}')

## Inertia: Within-Cluster Sum of Squares (WCSS)

### Plain English

For every point in the dataset, measure how far it is from its cluster's centroid. Square that distance (to penalise large errors more), then add up all those squared distances. That total is the **inertia**.

### Formula

$$\text{Inertia} = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

where $\mu_k$ is the centroid of cluster $k$.

### Interpretation

- Lower inertia → points are closer to their centroids → tighter clusters
- Inertia **always decreases** as *k* increases (more clusters = each point is closer to a centroid)
- This means you **cannot** just minimise inertia — otherwise you would always pick *k = n* (each point is its own cluster, inertia = 0)

### The Elbow Method

Plot inertia vs. *k* and look for the **elbow** — the point where adding more clusters gives diminishing returns. The elbow is a subjective heuristic; combine it with silhouette and DB for a more principled choice.

In [ ]:
np.random.seed(42)

def inertia_scratch(X, labels, centroids):
    """Compute WCSS (inertia) from scratch.
    
    Parameters
    ----------
    X        : (n, d) array of data points
    labels   : (n,) integer cluster assignment for each point
    centroids: (k, d) array of centroid coordinates
    
    Returns
    -------
    total inertia (float)
    """
    total = 0
    for k in range(len(centroids)):          # iterate over each cluster
        mask = labels == k                   # boolean mask: True for points in cluster k
        if mask.sum() > 0:                   # skip empty clusters (can happen with bad init)
            diff = X[mask] - centroids[k]    # (n_k, d): displacement from centroid
            total += np.sum(diff ** 2)       # sum of squared Euclidean distances
    return total

# Fit K-Means with k=4 so we have labels + centroids to test against
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
km4.fit(X_blobs)

scratch_val = inertia_scratch(X_blobs, km4.labels_, km4.cluster_centers_)
sklearn_val = km4.inertia_

print(f'Inertia (scratch) : {scratch_val:.4f}')
print(f'Inertia (sklearn) : {sklearn_val:.4f}')
print(f'Match: {np.isclose(scratch_val, sklearn_val, rtol=1e-5)}')

## Silhouette Score

### Plain English

For each individual point *i*, ask two questions:

1. **How cohesive is my cluster?** Compute *a(i)* = the average distance from point *i* to every other point in the **same** cluster. Small *a(i)* means the cluster is tight.
2. **How separated am I from the next-best cluster?** Compute *b(i)* = the average distance from point *i* to every point in the **nearest other** cluster. Large *b(i)* means good separation.

The silhouette value for point *i* is then:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))}$$

### Interpretation

| Value | Meaning |
|-------|---------|
| Close to **+1** | Point is well inside its own cluster and far from others |
| Close to **0** | Point is on the boundary between two clusters |
| Close to **−1** | Point is probably in the **wrong** cluster |

The **overall silhouette score** is the mean of all per-point values. Higher is better.

In [ ]:
np.random.seed(42)

def silhouette_score_scratch(X, labels):
    """Compute mean silhouette score from scratch (O(n^2) — readable, not optimised).
    
    Parameters
    ----------
    X      : (n, d) data array
    labels : (n,) cluster assignments
    
    Returns
    -------
    mean silhouette score (float in [-1, 1])
    """
    n = len(X)
    scores = np.zeros(n)               # per-point silhouette values
    unique_labels = np.unique(labels)
    
    for i in range(n):
        # --- a(i): mean distance to points in the SAME cluster ---
        same_cluster = X[labels == labels[i]]   # all points with same label
        if len(same_cluster) > 1:
            # compute Euclidean distance from point i to every same-cluster point
            dists_same = np.linalg.norm(same_cluster - X[i], axis=1)
            a = dists_same.mean()               # mean (includes self, which is 0 — tiny bias)
        else:
            a = 0                               # singleton cluster: no comparison possible
        
        # --- b(i): min mean distance to points in any OTHER cluster ---
        b = np.inf
        for k in unique_labels:
            if k == labels[i]:                  # skip the point's own cluster
                continue
            other_cluster = X[labels == k]
            mean_dist = np.mean(np.linalg.norm(other_cluster - X[i], axis=1))
            b = min(b, mean_dist)               # keep the nearest other cluster
        
        # --- s(i): combine a and b ---
        denom = max(a, b)
        scores[i] = (b - a) / denom if denom > 0 else 0
    
    return scores.mean()

# Quick verification on the k=4 solution
scratch_sil = silhouette_score_scratch(X_blobs, km4.labels_)
sklearn_sil = silhouette_score(X_blobs, km4.labels_)

print(f'Silhouette (scratch) : {scratch_sil:.4f}')
print(f'Silhouette (sklearn) : {sklearn_sil:.4f}')
print(f'Match (atol=0.01)   : {np.isclose(scratch_sil, sklearn_sil, atol=0.01)}')

In [ ]:
np.random.seed(42)

k_range = range(2, 7)    # test k from 2 to 6
sil_scores = []
km_models = {}

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blobs)
    sil = silhouette_score(X_blobs, labels)   # sklearn for speed
    sil_scores.append(sil)
    km_models[k] = km
    print(f'k={k}  silhouette={sil:.4f}')

# Plot silhouette score vs. k
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(k_range), sil_scores, marker='o', linewidth=2, color='steelblue')
ax.axvline(x=4, color='crimson', linestyle='--', label='True k=4')   # mark the true k
ax.set_xlabel('Number of clusters k')
ax.set_ylabel('Mean Silhouette Score')
ax.set_title('Silhouette Score vs. k  (peak should be at k=4)')
ax.legend()
plt.tight_layout()
plt.show()

best_k = list(k_range)[np.argmax(sil_scores)]
print(f'\nBest k by silhouette: {best_k}')

In [ ]:
np.random.seed(42)

def plot_silhouette_blade(X, labels, title='Silhouette Blade Plot'):
    """Create a per-sample silhouette plot (the 'blade' or 'knife' visualization).
    Each horizontal bar represents one data point; bars are sorted by cluster
    and then by silhouette value within each cluster.
    """
    n_clusters = len(np.unique(labels))
    sample_sil = silhouette_samples(X, labels)    # per-point silhouette values
    mean_sil = sample_sil.mean()
    
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = cm.tab10(np.linspace(0, 1, n_clusters))
    
    y_lower = 10   # starting y position for the first cluster
    for i, color in zip(range(n_clusters), colors):
        cluster_sil = np.sort(sample_sil[labels == i])   # sort for the blade shape
        cluster_size = len(cluster_sil)
        y_upper = y_lower + cluster_size
        
        # Fill horizontal bars from 0 to each silhouette value
        ax.fill_betweenx(
            np.arange(y_lower, y_upper),
            0, cluster_sil,
            facecolor=color, alpha=0.7,
            label=f'Cluster {i}'
        )
        # Label the cluster at the midpoint of its range
        ax.text(-0.05, y_lower + cluster_size / 2, str(i), fontsize=10)
        y_lower = y_upper + 10   # gap between clusters
    
    # Vertical dashed line at the mean silhouette — a good cluster has most bars past this
    ax.axvline(x=mean_sil, color='red', linestyle='--',
               label=f'Mean = {mean_sil:.3f}')
    ax.set_xlabel('Silhouette coefficient')
    ax.set_ylabel('Data points (grouped by cluster)')
    ax.set_title(title)
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

# Show the blade plot for k=4 (the correct k) vs k=3 (wrong k)
plot_silhouette_blade(X_blobs, km_models[4].labels_, title='Silhouette Blade: k=4')
plot_silhouette_blade(X_blobs, km_models[3].labels_, title='Silhouette Blade: k=3  (notice the wide negative region)')

## Davies-Bouldin Index

### Plain English

For each pair of clusters *(i, j)*, compute a **similarity ratio**:

$$R_{ij} = \frac{s_i + s_j}{d_{ij}}$$

where:
- $s_i$ = average distance of points in cluster *i* to its centroid (scatter, how spread-out the cluster is)
- $d_{ij}$ = distance between the centroids of clusters *i* and *j* (separation)

A large $R_{ij}$ means the two clusters are similar (bad). For each cluster *i*, find its **worst-case** similarity to any other cluster. The DB index averages these worst cases:

$$DB = \frac{1}{K} \sum_{i=1}^{K} \max_{j \neq i} R_{ij}$$

### Interpretation

- **Lower DB is better** (0 = perfect, completely non-overlapping clusters)
- DB penalises large within-cluster scatter and small between-cluster distances simultaneously
- It is more sensitive to cluster shape and size than silhouette

In [ ]:
np.random.seed(42)

def davies_bouldin_scratch(X, labels):
    """Compute Davies-Bouldin index from scratch.
    
    Parameters
    ----------
    X      : (n, d) data array
    labels : (n,) cluster assignments
    
    Returns
    -------
    DB index (float, lower = better)
    """
    unique_labels = np.unique(labels)
    k = len(unique_labels)
    
    # Compute centroid for each cluster
    centroids = np.array([
        X[labels == c].mean(axis=0) for c in unique_labels
    ])   # shape: (k, d)
    
    # s[i]: mean distance of points in cluster i to its centroid (scatter)
    s = np.array([
        np.mean(np.linalg.norm(X[labels == c] - centroids[i], axis=1))
        for i, c in enumerate(unique_labels)
    ])   # shape: (k,)
    
    db_sum = 0.0
    for i in range(k):
        worst_ratio = 0.0                   # find the worst (largest) R_ij for this cluster
        for j in range(k):
            if i == j:
                continue
            d_ij = np.linalg.norm(centroids[i] - centroids[j])   # centroid-to-centroid distance
            ratio = (s[i] + s[j]) / d_ij                         # similarity ratio R_ij
            worst_ratio = max(worst_ratio, ratio)                 # keep the worst case
        db_sum += worst_ratio
    
    return db_sum / k

# Verify against sklearn on the k=4 solution
scratch_db = davies_bouldin_scratch(X_blobs, km4.labels_)
sklearn_db = davies_bouldin_score(X_blobs, km4.labels_)

print(f'Davies-Bouldin (scratch) : {scratch_db:.4f}')
print(f'Davies-Bouldin (sklearn) : {sklearn_db:.4f}')
print(f'Match (atol=0.01)        : {np.isclose(scratch_db, sklearn_db, atol=0.01)}')

In [ ]:
np.random.seed(42)

db_scores = []
inertia_scores = []

for k in k_range:
    km = km_models[k]
    labels = km.labels_
    db_scores.append(davies_bouldin_score(X_blobs, labels))
    inertia_scores.append(km.inertia_)

# Side-by-side comparison of silhouette and DB index
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: silhouette (higher = better)
axes[0].plot(list(k_range), sil_scores, marker='o', color='steelblue', linewidth=2)
axes[0].axvline(x=4, color='crimson', linestyle='--', label='True k=4')
axes[0].set_title('Silhouette Score (↑ better)')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Score')
axes[0].legend()

# Right: Davies-Bouldin (lower = better)
axes[1].plot(list(k_range), db_scores, marker='s', color='darkorange', linewidth=2)
axes[1].axvline(x=4, color='crimson', linestyle='--', label='True k=4')
axes[1].set_title('Davies-Bouldin Index (↓ better)')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Score')
axes[1].legend()

plt.suptitle('Both metrics should agree on the optimal k', fontsize=12)
plt.tight_layout()
plt.show()

## Calinski-Harabasz Index

### Plain English

Also called the **Variance Ratio Criterion**, CH measures the ratio of:
- **Between-cluster dispersion:** how spread out the cluster centroids are from the overall mean
- **Within-cluster dispersion:** how spread out the points are within their clusters

$$CH = \frac{\text{Tr}(B_k) / (k-1)}{\text{Tr}(W_k) / (n-k)}$$

where $B_k$ is the between-cluster scatter matrix, $W_k$ is the within-cluster scatter matrix.

**Higher CH = better** (more separated and compact clusters).

CH tends to favour convex clusters and may over-penalise higher *k* on non-spherical data. Use it alongside silhouette and DB.

In [ ]:
np.random.seed(42)

ch_scores = [
    calinski_harabasz_score(X_blobs, km_models[k].labels_)
    for k in k_range
]

# Three subplots: inertia, silhouette, DB — all vs. k
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Inertia / Elbow
axes[0].plot(list(k_range), inertia_scores, marker='o', color='gray', linewidth=2)
axes[0].axvline(x=4, color='crimson', linestyle='--', label='True k=4')
axes[0].set_title('Inertia / Elbow (↓ better)')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].legend()

# 2. Silhouette
axes[1].plot(list(k_range), sil_scores, marker='o', color='steelblue', linewidth=2)
axes[1].axvline(x=4, color='crimson', linestyle='--', label='True k=4')
axes[1].set_title('Silhouette Score (↑ better)')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Score')
axes[1].legend()

# 3. Davies-Bouldin
axes[2].plot(list(k_range), db_scores, marker='s', color='darkorange', linewidth=2)
axes[2].axvline(x=4, color='crimson', linestyle='--', label='True k=4')
axes[2].set_title('Davies-Bouldin Index (↓ better)')
axes[2].set_xlabel('k')
axes[2].set_ylabel('Score')
axes[2].legend()

plt.suptitle('All three internal metrics point to k=4 on clean data', fontsize=12)
plt.tight_layout()
plt.show()

# Summary table
summary = pd.DataFrame({
    'k': list(k_range),
    'Inertia': [f'{v:.1f}' for v in inertia_scores],
    'Silhouette': [f'{v:.4f}' for v in sil_scores],
    'Davies-Bouldin': [f'{v:.4f}' for v in db_scores],
    'Calinski-Harabasz': [f'{v:.1f}' for v in ch_scores]
})
print(summary.to_string(index=False))

## External Metrics (When Ground Truth Is Available)

In real customer segmentation you never have true labels. But in research settings — or when you have a labelled validation set — you can use **external metrics** to measure how well your clusters recover the true groupings.

### Adjusted Rand Index (ARI)

Counts the number of point-pairs that are assigned to the **same cluster** by both the algorithm and the ground truth (and pairs assigned to **different** clusters by both). It adjusts for the agreement expected by chance.

- Range: −1 to +1. ARI = 1 means perfect agreement; ARI ≈ 0 means random assignment.

### Normalised Mutual Information (NMI)

Measures how much information the cluster labels and the true labels **share**, normalised to [0, 1]. NMI = 1 means the clustering perfectly reproduces the true groupings.

### When to Use External Metrics

- Benchmarking a new clustering algorithm on a public dataset with known labels
- Checking if an unsupervised approach can rediscover known categories (e.g., does K-Means on digits images find digit clusters?)
- **Never** use them to tune *k* in production — that would be label leakage

In [ ]:
np.random.seed(42)

# -----------------------------------------------------------------------
# Synthetic retail customer dataset
# Features: annual_spend, visit_frequency, avg_basket_size, loyalty_score, returns_rate
# -----------------------------------------------------------------------
n_customers = 300

# Cluster 0: High spenders, loyal, infrequent — premium shoppers
c0 = np.random.multivariate_normal(
    mean=[8000, 5, 120, 90, 2],
    cov=np.diag([500**2, 1**2, 15**2, 5**2, 1**2]),
    size=80
)
# Cluster 1: Low spenders, very frequent, small baskets — bargain hunters
c1 = np.random.multivariate_normal(
    mean=[1500, 20, 30, 40, 10],
    cov=np.diag([300**2, 4**2, 8**2, 10**2, 3**2]),
    size=100
)
# Cluster 2: Medium spenders, moderate frequency — mainstream shoppers
c2 = np.random.multivariate_normal(
    mean=[3500, 10, 65, 60, 5],
    cov=np.diag([400**2, 2**2, 10**2, 8**2, 2**2]),
    size=120
)

X_retail = np.vstack([c0, c1, c2])
y_retail = np.array([0]*80 + [1]*100 + [2]*120)  # true labels (for external metrics only)

# Scale features — essential when features have different units
scaler = StandardScaler()
X_retail_scaled = scaler.fit_transform(X_retail)

feature_names = ['annual_spend', 'visit_freq', 'avg_basket', 'loyalty', 'returns']
print(f'Retail dataset shape: {X_retail_scaled.shape}')
print(f'True cluster counts:  {np.bincount(y_retail)}')

# -----------------------------------------------------------------------
# Evaluate all metrics for k = 2, 3, 4, 5
# -----------------------------------------------------------------------
results = []

for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    preds = km.fit_predict(X_retail_scaled)
    
    row = {
        'k': k,
        'Inertia': round(km.inertia_, 1),
        'Silhouette': round(silhouette_score(X_retail_scaled, preds), 4),
        'Davies-Bouldin': round(davies_bouldin_score(X_retail_scaled, preds), 4),
        'Calinski-Harabasz': round(calinski_harabasz_score(X_retail_scaled, preds), 1),
        'ARI': round(adjusted_rand_score(y_retail, preds), 4),
        'NMI': round(normalized_mutual_info_score(y_retail, preds), 4),
    }
    results.append(row)

df_results = pd.DataFrame(results)

# Mark the row with the best silhouette score
best_k_idx = df_results['Silhouette'].idxmax()
df_results['Recommended'] = ''
df_results.loc[best_k_idx, 'Recommended'] = '<<< BEST'

print('\nMetric Summary Table (retail data):')
print(df_results.to_string(index=False))

In [ ]:
np.random.seed(42)

# Fit K-Means with the recommended k on retail data
best_k_retail = int(df_results.loc[df_results['Recommended'] == '<<< BEST', 'k'].values[0])
print(f'Recommended k: {best_k_retail}')

km_retail = KMeans(n_clusters=best_k_retail, random_state=42, n_init=10)
retail_labels = km_retail.fit_predict(X_retail_scaled)

# Silhouette blade plot for the recommended k
plot_silhouette_blade(
    X_retail_scaled, retail_labels,
    title=f'Silhouette Blade Plot — Retail Data, k={best_k_retail}'
)

# Also visualise the clusters on the first two features
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    X_retail_scaled[:, 0], X_retail_scaled[:, 1],
    c=retail_labels, cmap='tab10', s=30, alpha=0.7
)
ax.set_xlabel('Annual Spend (scaled)')
ax.set_ylabel('Visit Frequency (scaled)')
ax.set_title(f'Retail Customer Segments (k={best_k_retail})')
plt.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.show()

## Self-Check Questions

Test your understanding before moving on. Try to answer each question yourself first.

---

**Question 1:** Inertia always decreases as *k* increases. Does that mean we should just pick the largest *k* possible to minimise inertia?

<details><summary>Show answer</summary>

No. At *k = n* (every point is its own cluster), inertia = 0 — but that is completely useless for any analytical purpose. Inertia is not a standalone optimisation target; it is used to find the **elbow** where the rate of decrease flattens out. Beyond the elbow, you are adding clusters that provide no meaningful new grouping. Always combine the elbow with silhouette and DB.

</details>

---

**Question 2:** A data point has silhouette value −0.3. What does this tell you, and what might you do about it?

<details><summary>Show answer</summary>

A silhouette of −0.3 means the point is **closer on average to a neighbouring cluster** than to its own cluster. It has likely been assigned to the wrong cluster. In practice you might: (a) try a different random initialisation, (b) increase *k*, (c) check if the point is a genuine outlier/noise point, or (d) switch to a density-based method like DBSCAN that can label borderline points as noise rather than forcing them into a cluster.

</details>

---

**Question 3:** You are segmenting customers for a marketing campaign. You have no ground-truth labels. Your colleague says "let's just use ARI to evaluate our clusters." What would you say?

<details><summary>Show answer</summary>

ARI (Adjusted Rand Index) is an **external** metric — it requires ground-truth labels to compare against. If you have no true labels, ARI is not computable. Instead use internal metrics: silhouette score (how tight and separated the clusters are), Davies-Bouldin index (worst-case cluster similarity), and the elbow plot (to sanity-check inertia). If you later obtain labels (e.g., from a small pilot campaign), you can then also compute ARI as a retrospective validation.

</details>

## Key Takeaways

1. **Never rely on a single metric.** Use the elbow (inertia), silhouette score, and Davies-Bouldin together. If they agree on *k*, you can be confident. If they disagree, investigate why.

2. **Inertia is an elbow heuristic, not an optimisation target.** It always decreases with *k*; look for the point of diminishing returns.

3. **Silhouette score is the most interpretable internal metric.** Range −1 to +1; higher is better. The blade plot reveals individual cluster quality — wide blades above the mean are healthy clusters.

4. **Davies-Bouldin penalises similar clusters.** It explicitly models the worst-case confusion between any pair of clusters, making it useful when clusters have similar densities.

5. **External metrics (ARI, NMI) require labels.** In real deployments you will not have them. Reserve external metrics for research benchmarks or retrospective validation.

6. **Scaling matters.** Always standardise features before clustering and before computing any distance-based metric.

---

**Next notebook:** `08_PCA.ipynb` — Principal Component Analysis for dimensionality reduction and visualisation.